<a href="https://colab.research.google.com/github/oliveira-o2/AMORA/blob/main/AMORA_BASE_CONSOLIDADA.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [5]:
def consolidar_arquivos():
    print("Buscando arquivos...")
    nome_saida = "Relatório de Itens por NFe - Consolidado.xlsx"

    # 1. Buscamos os arquivos, mas IGNORAMOS o arquivo consolidado se ele já existir
    arquivos = [f for f in (glob.glob("Relatório de Itens por NFe*.xls") +
                            glob.glob("Relatório de Itens por NFe*.xlsx"))
                if f != nome_saida]

    if not arquivos:
        print("Arquivos não encontrados na pasta atual.")
        return

    lista_df = []

    for f in arquivos:
        try:
            df = ler_excel(f)
            df = df[~df.iloc[:, 0].astype(str).str.contains("Total|Soma|Somatório", case=False, na=False)]
            colunas_c_em_diante = df.columns[2:]
            df = df.dropna(subset=colunas_c_em_diante, how='all')

            lista_df.append(df)
            print(f"✔ Arquivo processado: {f}")
        except Exception as e:
            print(f"❌ Erro ao processar o arquivo {f}: {e}")

    if not lista_df:
        print("Nenhum dado pôde ser extraído após a limpeza.")
        return

    print("\nConsolidando os dados...")
    df_final = pd.concat(lista_df, ignore_index=True)

    # Conversão de tipos
    df_final.iloc[:, 0] = pd.to_datetime(df_final.iloc[:, 0], errors='coerce', dayfirst=True)
    colunas_para_numero = [4] + list(range(12, 23))
    for col_idx in colunas_para_numero:
        if col_idx < len(df_final.columns):
            df_final.iloc[:, col_idx] = pd.to_numeric(df_final.iloc[:, col_idx], errors='coerce')

    # --- Criação da aba CFOP ---
    cfops_unicos = df_final.iloc[:, 10].unique() if len(df_final.columns) > 10 else []
    df_cfop = pd.DataFrame(cfops_unicos, columns=['CFOP'])
    df_cfop['Descrição'] = ""
    df_cfop['Tipo'] = ""

    # Regras de Classificação baseadas nos CFOPs fornecidos
    vendas = [5101, 6101, 5102, 6102, 6152, '5101', '6101', '5102', '6102', '6152']
    devolucoes = [1201, 1202, 2201, 2202, 5202, 6202, '1201', '1202', '2201', '2202', '5202', '6202']
    bonificacao = [5910, 6910, '5910', '6910']
    remessas = [5411, 5911, 6911, 5949, 5405, '5411', '5911', '6911', '5949', '5405', '59505']
    ajustes = [5927, '5927']

    df_cfop.loc[df_cfop['CFOP'].isin(vendas), 'Tipo'] = 'Venda'
    df_cfop.loc[df_cfop['CFOP'].isin(devolucoes), 'Tipo'] = 'Devolução'
    df_cfop.loc[df_cfop['CFOP'].isin(bonificacao), 'Tipo'] = 'Bonificação'
    df_cfop.loc[df_cfop['CFOP'].isin(remessas), 'Tipo'] = 'Remessa/Outros'
    df_cfop.loc[df_cfop['CFOP'].isin(ajustes), 'Tipo'] = 'Ajuste/Perda'

    try:
        with pd.ExcelWriter(nome_saida, engine='xlsxwriter') as writer:
            df_final.to_excel(writer, sheet_name='Consolidado', index=False)
            df_cfop.to_excel(writer, sheet_name='CFOP', index=False)

        print("Aplicando formatação no Excel...")
        formatar_excel(nome_saida)
        print(f"\n✅ Sucesso! Arquivo gerado com abas Consolidado e CFOP: '{nome_saida}'")
    except PermissionError:
        print(f"\n❌ ERRO: O arquivo '{nome_saida}' está aberto no Excel.")
        print("Por favor, feche o arquivo e execute o script novamente.")

In [6]:
consolidar_arquivos()

Buscando arquivos...
Arquivos não encontrados na pasta atual.


In [7]:
import os
import glob

nome_arquivo = 'Relatório de Itens por NFe - Consolidado.xlsx'
exists = os.path.exists(nome_arquivo)

if exists:
    print(f"✅ O arquivo '{nome_arquivo}' foi encontrado!")
    stats = os.stat(nome_arquivo)
    print(f"Tamanho: {stats.st_size / 1024:.2f} KB")
else:
    print(f"❌ O arquivo '{nome_arquivo}' NÃO foi encontrado.")
    print("Arquivos Excel disponíveis na pasta:")
    print(glob.glob('*.xls*'))

❌ O arquivo 'Relatório de Itens por NFe - Consolidado.xlsx' NÃO foi encontrado.
Arquivos Excel disponíveis na pasta:
[]


In [8]:
import pandas as pd
import glob

# Primeiro, garantimos que o arquivo seja gerado
consolidar_arquivos()

def verificar_cfops_mapeados():
    nome_saida = 'Relatório de Itens por NFe - Consolidado.xlsx'
    try:
        # Lendo a aba CFOP do arquivo gerado
        df_cfop = pd.read_excel(nome_saida, sheet_name='CFOP')

        # Identificando CFOPs sem 'Tipo'
        # Garantindo que comparamos com string vazia ou NaN
        nao_mapeados = df_cfop[df_cfop['Tipo'].isna() | (df_cfop['Tipo'].astype(str).str.strip() == '')]

        if not nao_mapeados.empty:
            print('⚠️ CFOPs encontrados que não possuem mapeamento:')
            display(nao_mapeados[['CFOP']])
        else:
            print('✅ Todos os CFOPs encontrados foram devidamente mapeados!')

        print('\nResumo de CFOPs por Tipo:')
        display(df_cfop['Tipo'].value_counts(dropna=False))

    except Exception as e:
        print(f'Erro ao verificar o arquivo: {e}')

verificar_cfops_mapeados()

Buscando arquivos...
Arquivos não encontrados na pasta atual.
Erro ao verificar o arquivo: [Errno 2] No such file or directory: 'Relatório de Itens por NFe - Consolidado.xlsx'


In [ ]:
import os

caminho_atual = os.getcwd()
print(f"Diretório atual de busca: {caminho_atual}")

print("\nConteúdo da pasta:")
arquivos = os.listdir(caminho_atual)
if arquivos:
    for arquivo in arquivos:
        print(f"- {arquivo}")
else:
    print("A pasta está vazia.")